# 14 · Sparse feature discovery in many dimensions

A controlled benchmark: a hidden law `y = 3 x₁² - 2 x₂x₃ + sin(x₄) + noise`
lives inside a wide, mostly-irrelevant feature space. We fit a smooth omnibias
field, screen closed-form operator features on the **training split only**, and
recover the active terms — then show the discovered features make a linear model
competitive with an exhaustive dictionary while using far fewer columns.

Source: `examples/symbolic_discovery/synthetic_feature_discovery/`.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "..")  # so `examples.*` is importable from the repo root
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from examples.symbolic_discovery.synthetic_feature_discovery.benchmark import evaluate_benchmark

res = evaluate_benchmark(n_samples=2500, noise_std=0.1, hidden=400, seed=0)
print("hidden law:", res["hidden_law"])
print("field train RMSE:", f"{res['field_train_rmse']:.4f}")

## Discovered features

Train-only screening ranks candidate closed-form features by correlation with the
target. The top picks are exactly the hidden terms.

In [ ]:
feats = res["discovered_features"]
for row in feats:
    print(f"  {row['name']:<10}  score={row['score']:.3f}")

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.bar([r["name"] for r in feats], [r["score"] for r in feats], color=PRIMARY)
ax.set_ylabel("screening score")
ax.set_title("Recovered active features (x₁², x₂x₃, sin x₄)")
plt.tight_layout()

## Model comparison

A linear model on the few discovered features matches a 22-column exhaustive
dictionary and a gradient-boosting model on raw inputs — at a fraction of the
complexity.

In [ ]:
models = res["models"]
order = sorted(models.items(), key=lambda kv: kv[1]["rmse"])
names = [k for k, _ in order]
rmses = [v["rmse"] for _, v in order]
fig, ax = plt.subplots(figsize=(8.0, 4.0))
colors = [GOOD if "omnibias" in n else PRIMARY for n in names]
ax.barh(names, rmses, color=colors)
ax.set_xscale("log")
ax.set_xlabel("test RMSE (log)")
ax.set_title("omnibias discovered features vs baselines")
plt.tight_layout()
for k, v in order:
    print(f"  {k:<48} RMSE={v['rmse']:.3f}  (n_features={v.get('n_features','?')})")

## Takeaway

Closed-form operator features turn a black-box regression into an interpretable
one: the discovered three-term model is both accurate and readable, while raw
linear regression is far off. The real-data analogue (NASA C-MAPSS turbofan RUL)
is in notebook 16.